# wav2vec2 Fine-Tuning

This notebook runs the main `facebook/wav2vec2-base` fine-tuning pipeline for the project.

It supports both:

- local execution with the project `.venv`
- a Colab-backed kernel from VS Code or Colab itself

Important note for Colab-backed execution:

The remote runtime must actually have a copy of this project folder. The notebook will search common locations such as `/content/FinalProject` and Google Drive after mounting it.


In [1]:
from __future__ import annotations

import importlib.util
import os
import subprocess
import sys
from pathlib import Path


def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False


IS_COLAB = running_in_colab()
print('Running in Colab:', IS_COLAB)

if IS_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd.parent,
        cwd / 'FinalProject',
        Path('/content/FinalProject'),
        Path('/content/drive/MyDrive/FinalProject'),
        Path('/content/drive/MyDrive/Colab Notebooks/FinalProject'),
    ]

    for base in [Path('/content'), Path('/content/drive/MyDrive'), Path('/content/drive/MyDrive/Colab Notebooks')]:
        if base.exists():
            for child in base.glob('*'):
                candidates.append(child)

    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / 'configs' / 'wav2vec.yaml').exists() and (candidate / 'src').exists():
            return candidate

    raise FileNotFoundError(
        'Could not find the project root in the current runtime. '
        'Make sure the FinalProject folder exists in the local workspace or Colab runtime before running this notebook.'
    )


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)
print('Working directory:', Path.cwd().resolve())

if IS_COLAB:
    package_map = {
        'yaml': 'pyyaml',
        'pandas': 'pandas',
        'numpy': 'numpy',
        'soundfile': 'soundfile',
        'librosa': 'librosa',
        'torch': 'torch',
        'torchaudio': 'torchaudio',
        'transformers': 'transformers',
        'datasets': 'datasets',
        'accelerate': 'accelerate',
        'sklearn': 'scikit-learn',
        'tqdm': 'tqdm',
    }
    missing_packages = sorted({pkg for module, pkg in package_map.items() if importlib.util.find_spec(module) is None})
    if missing_packages:
        print('Installing missing packages:', missing_packages)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing_packages])
    else:
        print('No missing packages detected for Colab runtime.')


Mounted at /content/drive


FileNotFoundError: Could not find the project root in the current runtime. Make sure the FinalProject folder exists in the local workspace or Colab runtime before running this notebook.

In [ ]:
import json

from src.training.train_wav2vec import run_training
from src.utils.config import load_yaml_config

config_path = PROJECT_ROOT / 'configs' / 'wav2vec.yaml'
config = load_yaml_config(config_path)

metadata_path = (PROJECT_ROOT / config['metadata_path']).resolve()
output_dir = (PROJECT_ROOT / config['output_dir']).resolve()

assert config_path.exists(), f'Config file not found: {config_path}'
assert metadata_path.exists(), f'Metadata file not found: {metadata_path}'

print(json.dumps(config, indent=2))


In [ ]:
print('Config file:', config_path)
print('Metadata file:', metadata_path)
print('Output directory:', output_dir)
print('Backbone:', config['backbone_name'])
print('Epochs:', config['num_epochs'])
print('Batch size:', config['batch_size'])
print('Eval batch size:', config['eval_batch_size'])
print('Learning rate:', config['learning_rate'])
print('Sample rate:', config['sample_rate'])


In [ ]:
run_training(config, dry_run=False)


In [ ]:
print('Output directory:', output_dir)
print('Exists:', output_dir.exists())

if output_dir.exists():
    for path in sorted(output_dir.iterdir()):
        print(path.name)
